# Track B: reinforcement learning

Train a policy on the model you identified in steps 1 to 3, then fly it on the drone in MATLAB.

1. Write a Gymnasium environment around your model (like `RocketLanderEnv` in TD 4). On the drone, the
   controller measures at each step ($T_s = 0.05$ s): `t`, `z`, `vz`, `az_prev`, `z_ref`, `u_prev`
   (`az_prev`, `u_prev`: acceleration and command of the previous step), and sends $u \in [-1, 1]$.
   `reset()` and `step()` return them in `info["m"]`.
2. Train a Stable-Baselines3 agent with an `MlpPolicy` on it, then export it with
   `export_for_matlab(agent, env)` (`export_policy.py`, do not edit): it writes `policy.mat` and `traj_model.mat`.
   Docs: [Stable-Baselines3](https://stable-baselines3.readthedocs.io/), and to follow the training,
   [TensorBoard with SB3](https://stable-baselines3.readthedocs.io/en/master/guide/tensorboard.html).
3. Write the same mission and observation in `mission()` and `observation()` of `rl_controller.m`, then run
   `EINT_RL_eval` in MATLAB. Complete the TODO in the matlab files to run the mission.

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from export_policy import export_for_matlab
from Env_drone import DroneEnv

# PPO avec 16 vols en parallèle : ~50 fois plus rapide que SAC sur ce modèle (500k pas en quelques minutes)
envs = make_vec_env(DroneEnv, n_envs=16, seed=0)
agent = PPO("MlpPolicy", envs, verbose=1, seed=0, n_steps=256, batch_size=512,
            gamma=0.99, gae_lambda=0.95, policy_kwargs=dict(log_std_init=-1.0))
agent.learn(total_timesteps=500_000)

env = DroneEnv()
export_for_matlab(agent, env)   # env without noise: writes policy.mat and traj_model.mat